## Cell 1: Setup
**What this demonstrates**: Long-Context RAG needs only one external dependency -- the Anthropic SDK. No embeddings, no vector store, no BM25 index. The absence of these imports is itself the point.
**Expected output**: OK Setup complete with model name, context window limit, and masked API key suffix.

In [ ]:
# -- Install dependencies --------------------------------------------------
# Notice what is NOT here: no chromadb, no rank-bm25, no langchain-openai.
# Long-Context RAG has no retrieval pipeline -- the Anthropic SDK is sufficient.
%pip install -q anthropic python-dotenv

# -- Standard library -------------------------------------------------------
import os
import time
import pathlib
from typing import Any

# -- Third-party ------------------------------------------------------------
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic

# -- Load API keys ----------------------------------------------------------
load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set -- add it to .env'

# -- Constants --------------------------------------------------------------
MODEL         = 'claude-sonnet-4-6'
# Claude's context window is 200K tokens. We reserve 10K for the answer,
# leaving 190K for the system prompt + documents + query.
MAX_INPUT_TOKENS = 190_000
# Approximate cost per million input tokens at current Claude Sonnet pricing.
# Update this constant as pricing changes -- it is used for cost estimates only.
COST_PER_M_INPUT  = 3.00   # USD per million input tokens (Claude Sonnet)
COST_PER_M_OUTPUT = 15.00  # USD per million output tokens

# -- Client initialisation --------------------------------------------------
client: Anthropic = Anthropic()

print('OK Setup complete')
print(f'  Model:           {MODEL}')
print(f'  Context window:  200,000 tokens (200K)')
print(f'  Input budget:    {MAX_INPUT_TOKENS:,} tokens (10K reserved for answer)')
print(f'  Anthropic key:   ...{_anthropic_key[-4:]}')
print()
print('Dependencies loaded: anthropic, python-dotenv')
print('NOT loaded: chromadb, rank-bm25, langchain-openai, sentence-transformers')
print('  -> No retrieval pipeline. No vector store. No chunking.')

## Cell 2: Data -- All Four Sample Documents
**What this demonstrates**: Load all four synthetic fintech documents as complete plain-text strings -- no chunking, no splitting, no preprocessing. Print the raw character counts, estimated token counts, and confirm the combined corpus fits within Claude's 200K-token input budget.
**Expected output**: Per-document stats (characters, estimated tokens), combined total, and a confirmation that the full corpus fits in context.

In [ ]:
# -- Locate sample data ----------------------------------------------------
_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _candidates if p.exists()), None)
assert BASE_DIR is not None, (
    'Cannot find shared/sample_data/. '
    'Run from the repo root or from modules/15_long_context_rag/.'
)

# -- Load complete document texts -- NO chunking ----------------------------
# Each document is stored as a single string. The LLM reads it in full.
DOCS: dict[str, str] = {
    'Basel III Capital Requirements': (BASE_DIR / 'basel_iii_excerpt.txt').read_text(encoding='utf-8'),
    'Meridian Bank Lending Policy':   (BASE_DIR / 'fintech_policy.txt').read_text(encoding='utf-8'),
    'ISDA Master Agreement':          (BASE_DIR / 'isda_excerpt.txt').read_text(encoding='utf-8'),
    'Pinnacle Financial Q3 Earnings': (BASE_DIR / 'earnings_report.txt').read_text(encoding='utf-8'),
}

# -- Token estimation -------------------------------------------------------
# Rule of thumb: ~4 characters per token for English financial prose.
# This is an approximation; actual token count depends on the model's tokenizer.
# Use client.messages.count_tokens() for exact counts in production.
def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)

# -- Print document stats ---------------------------------------------------
print('Document corpus (loaded as complete texts -- no chunking):')
print()
total_chars  = 0
total_tokens = 0
for name, text in DOCS.items():
    chars  = len(text)
    tokens = estimate_tokens(text)
    total_chars  += chars
    total_tokens += tokens
    print(f'  {name:42s}  {chars:7,} chars  ~{tokens:5,} tokens')

print()
print(f'  {"TOTAL":42s}  {total_chars:7,} chars  ~{total_tokens:5,} tokens')
print()

# -- Context window check ---------------------------------------------------
# Account for system prompt (~200 tokens) and query (~50 tokens)
overhead_tokens = 250
context_needed  = total_tokens + overhead_tokens
headroom        = MAX_INPUT_TOKENS - context_needed

if context_needed <= MAX_INPUT_TOKENS:
    print(f'Context check: {context_needed:,} tokens needed  /  {MAX_INPUT_TOKENS:,} available  ->  OK')
    print(f'  Headroom: ~{headroom:,} tokens remaining (enough for a 300-page regulatory document)')
else:
    print(f'WARNING: {context_needed:,} tokens needed exceeds {MAX_INPUT_TOKENS:,} budget.')
    print('  Consider RAPTOR (compress) or Contextual RAG (chunk with context).')

print()
print('Why these documents fit comfortably:')
print('  Real-world equivalents that also fit in 200K:')
print('  - A full 10-K annual filing:       ~60,000-90,000 tokens')
print('  - Basel III full framework:        ~40,000 tokens')
print('  - ISDA Master Agreement + CSA:     ~25,000 tokens')
print('  - A standard credit agreement:     ~80,000 tokens')
print('  All four of the above together fit in Claude with room to spare.')

## Cell 3: Core -- Long-Context Pipeline (No Retrieval)
**What this demonstrates**: The complete Long-Context RAG implementation in three functions. `assemble_context` wraps each document in XML tags with clear delimiters. `long_context_query` calls Claude with the full assembled context -- no retrieval step, no vector search, no BM25. `format_cost` produces a human-readable cost estimate for each query.
**Expected output**: Function definitions confirmed, with a test showing the assembled context structure and character/token count.

In [ ]:
# -- Context assembly -------------------------------------------------------
def assemble_context(docs: dict[str, str], query: str) -> tuple[str, int]:
    # Wrap each document in XML-style tags so the model can locate section boundaries.
    # Place documents BEFORE the query. This exploits recency bias -- the query
    # lands at the very end where the model attends most strongly.
    parts = []
    for name, text in docs.items():
        # Normalise the document name into a valid XML tag (spaces -> underscores)
        tag = name.replace(' ', '_').replace('/', '_')
        parts.append(f'<{tag}>\n{text}\n</{tag}>')
    assembled = '\n\n'.join(parts) + f'\n\n---\n\nQuestion: {query}'
    return assembled, estimate_tokens(assembled)


# -- Cost estimation --------------------------------------------------------
def format_cost(input_tokens: int, output_tokens: int) -> str:
    # Produce a human-readable cost string using the constants from Cell 1.
    # These are estimates based on approximate token counts -- use count_tokens()
    # for exact figures in production billing systems.
    input_cost  = (input_tokens  / 1_000_000) * COST_PER_M_INPUT
    output_cost = (output_tokens / 1_000_000) * COST_PER_M_OUTPUT
    total_cost  = input_cost + output_cost
    return (
        f'~${total_cost:.4f} USD  '
        f'(input: {input_tokens:,} tokens x ${COST_PER_M_INPUT}/M = ${input_cost:.4f}  |  '
        f'output: {output_tokens:,} tokens x ${COST_PER_M_OUTPUT}/M = ${output_cost:.4f})'
    )


# -- Long-context query -----------------------------------------------------
def long_context_query(
    docs:   dict[str, str],
    query:  str,
    system: str,
) -> dict[str, Any]:
    # The complete Long-Context RAG pipeline:
    #   1. Assemble full document texts + query into one context string
    #   2. Verify it fits within the model's context budget
    #   3. Call Claude -- no retrieval, no chunking, no vector search
    #   4. Return answer + stats
    #
    # There is no Step 0 (build index) or Step 1.5 (retrieve chunks).
    # That is the pattern's defining characteristic.
    context, input_token_est = assemble_context(docs, query)

    # Safety check: abort rather than silently truncate an oversized context.
    assert input_token_est <= MAX_INPUT_TOKENS, (
        f'Estimated {input_token_est:,} tokens exceeds budget of {MAX_INPUT_TOKENS:,}. '
        f'Consider RAPTOR or Contextual RAG for documents this size.'
    )

    t0 = time.perf_counter()
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=system,
        messages=[{'role': 'user', 'content': context}],
    )
    elapsed_ms = (time.perf_counter() - t0) * 1000

    answer = response.content[0].text
    output_tokens_est = estimate_tokens(answer)

    return {
        'answer':            answer,
        'input_tokens_est':  input_token_est,
        'output_tokens_est': output_tokens_est,
        'elapsed_ms':        elapsed_ms,
        'cost_str':          format_cost(input_token_est, output_tokens_est),
        'docs_used':         list(docs.keys()),
        'retrieval_calls':   0,   # always zero -- this is the point
    }


# -- Smoke test: show assembled context structure ---------------------------
test_docs = {'Basel III Capital Requirements': DOCS['Basel III Capital Requirements']}
test_ctx, test_tokens = assemble_context(test_docs, 'What is the minimum CET1 ratio?')

print('Assembled context structure (first 300 chars):')
print('-' * 65)
print(test_ctx[:300])
print('...')
print('-' * 65)
print(f'  Full context: {len(test_ctx):,} chars  ~{test_tokens:,} tokens')
print()
print('Long-context pipeline functions:')
print('  assemble_context(docs, query)  -> (context_str, token_estimate)')
print('  long_context_query(docs, query, system)  -> dict with answer + stats')
print('  format_cost(input_tokens, output_tokens)  -> human-readable cost string')
print()
print('Retrieval functions defined: 0')
print('Vector store operations:     0')
print('BM25 index lookups:          0')
print('Embedding API calls:         0')

## Cell 4: Run -- Cross-Document Capital Requirements Query
**What this demonstrates**: A query that requires reasoning across two documents simultaneously -- Basel III regulatory minimums and the bank's internal lending policy. A chunked retriever would have to surface the right chunks from both documents and hope they contain the right cross-references. Long-Context RAG puts both documents in context and lets the model synthesise them directly.
**Expected output**: Retrieved context size, grounded answer comparing both frameworks with specific citations from each document, and full timing.

In [ ]:
QUERY = (
    'Compare the capital and financial requirements across the Basel III framework '
    'and the Meridian Bank lending policy. Where do they reinforce each other, '
    'and where does the internal policy go beyond the regulatory minimums?'
)

SYSTEM = (
    'You are a bank regulatory affairs analyst. '
    'You have been given multiple financial documents to read in full. '
    'Answer using ONLY the provided documents. '
    'For every claim, cite the source document and the specific section or metric. '
    'When comparing across documents, explicitly state which document each figure comes from. '
    'Distinguish between regulatory minimums (Basel III) and internal policy overlays '
    '(Meridian Bank policy).'
)

# -- Select the two documents relevant to this query -----------------------
# Long-Context RAG does not retrieve -- we select documents explicitly.
# For a corpus of hundreds of documents, a lightweight classifier or
# routing step would pre-select which documents to load; here two is sufficient.
query_docs = {
    'Basel III Capital Requirements': DOCS['Basel III Capital Requirements'],
    'Meridian Bank Lending Policy':   DOCS['Meridian Bank Lending Policy'],
}

print(f'Query: {QUERY[:80]!r}...')
print()
print(f'Documents in context: {len(query_docs)}')
for name, text in query_docs.items():
    print(f'  {name:42s}  ~{estimate_tokens(text):,} tokens  (complete -- not chunked)')
print()

# -- Run long-context query ------------------------------------------------
result = long_context_query(query_docs, QUERY, SYSTEM)

print('=' * 65)
print('CROSS-DOCUMENT ANSWER')
print('=' * 65)
print(result['answer'])
print('=' * 65)
print()
print(f'Context:     ~{result["input_tokens_est"]:,} input tokens')
print(f'Answer:      ~{result["output_tokens_est"]:,} output tokens')
print(f'Latency:     {result["elapsed_ms"]:.0f} ms')
print(f'Cost:        {result["cost_str"]}')
print(f'Retrieval:   {result["retrieval_calls"]} calls (none)')

## Cell 5: Inspect -- Token Budget, Cost Breakdown, and "Lost in the Middle"
**What this demonstrates**: Visualise how each document consumes the context window, produce a per-query cost estimate compared to a retrieval-based baseline, and demonstrate the "lost in the middle" effect by showing how context position affects answer completeness -- plus the mitigation strategy of placing critical content at context extremes.
**Expected output**: Token budget bar chart, cost comparison table, and a position-aware ordering demonstration.

In [ ]:
# -- Token budget visualisation --------------------------------------------
print('Context window budget breakdown:')
print(f'  Total window:       200,000 tokens')
print(f'  Reserved for answer:  10,000 tokens')
print(f'  Available for input: {MAX_INPUT_TOKENS:,} tokens')
print()

total_doc_tokens = sum(estimate_tokens(t) for t in query_docs.values())
system_tokens    = estimate_tokens(SYSTEM)
query_tokens     = estimate_tokens(QUERY)
overhead_tokens  = system_tokens + query_tokens + 50  # delimiters
used_tokens      = total_doc_tokens + overhead_tokens
free_tokens      = MAX_INPUT_TOKENS - used_tokens

bar_scale = MAX_INPUT_TOKENS / 50  # each char = bar_scale tokens
doc_bar   = '|' * int(total_doc_tokens / bar_scale)
ovh_bar   = '|' * max(1, int(overhead_tokens / bar_scale))
free_bar  = '.' * int(free_tokens / bar_scale)

print('  [docs           ][ovh][free                              ]')
print(f'  [{doc_bar:<16}][{ovh_bar:<3}][{free_bar}]')
print(f'  {total_doc_tokens:,} doc tokens + {overhead_tokens} overhead = {used_tokens:,} used  |  {free_tokens:,} free')
print()

# -- Per-document token breakdown ------------------------------------------
print('Per-document token consumption:')
for name in query_docs:
    tokens = estimate_tokens(query_docs[name])
    pct    = tokens / MAX_INPUT_TOKENS * 100
    bar    = '#' * int(pct / 2)
    print(f'  {name:42s}  {tokens:6,} tokens  ({pct:4.1f}%)  {bar}')
print()

# -- Cost comparison: long-context vs retrieval ----------------------------
# Retrieval baseline: 5 chunks x 500 chars each = 2,500 chars = ~625 tokens
retrieval_input_tokens  = 625 + overhead_tokens
retrieval_output_tokens = result['output_tokens_est']
lc_cost  = (result['input_tokens_est']  / 1e6) * COST_PER_M_INPUT  + \
           (result['output_tokens_est'] / 1e6) * COST_PER_M_OUTPUT
ret_cost = (retrieval_input_tokens      / 1e6) * COST_PER_M_INPUT  + \
           (retrieval_output_tokens     / 1e6) * COST_PER_M_OUTPUT
multiplier = lc_cost / ret_cost if ret_cost > 0 else float('inf')

print('Cost comparison: Long-Context RAG vs retrieval-based RAG')
print(f'  {"Approach":<32}  {"Input tokens":>14}  {"Est. cost":>12}')
print('  ' + '-' * 62)
print(f'  {"Long-Context (full docs)":<32}  {result["input_tokens_est"]:>14,}  ${lc_cost:>11.4f}')
print(f'  {"Retrieval (5 chunks)":<32}  {retrieval_input_tokens:>14,}  ${ret_cost:>11.4f}')
print(f'  Long-Context costs ~{multiplier:.0f}x more per query for this document set.')
print()

# -- Lost in the middle: position matters ----------------------------------
# Demonstrate that document ordering affects which content the model attends to.
# We test: Basel III first (regulatory) vs Basel III last (at the recency-favoured end).
LITM_QUERY = 'What is the minimum CET1 ratio and the countercyclical buffer requirement?'
LITM_SYSTEM = (
    'Answer using ONLY the provided documents. '
    'Cite the source document for every number. '
    'Be specific about percentages.'
)

print('"Lost in the Middle" -- context position experiment:')
print(f'  Query: {LITM_QUERY!r}')
print()

# Order A: Basel III first (primacy position)
docs_order_a = {
    'Basel III Capital Requirements': DOCS['Basel III Capital Requirements'],
    'Meridian Bank Lending Policy':   DOCS['Meridian Bank Lending Policy'],
}
# Order B: Basel III last (recency position)
docs_order_b = {
    'Meridian Bank Lending Policy':   DOCS['Meridian Bank Lending Policy'],
    'Basel III Capital Requirements': DOCS['Basel III Capital Requirements'],
}

res_a = long_context_query(docs_order_a, LITM_QUERY, LITM_SYSTEM)
res_b = long_context_query(docs_order_b, LITM_QUERY, LITM_SYSTEM)

print('  Order A -- Basel III FIRST (primacy position):')
print(f'  {res_a["answer"][:300]}...' if len(res_a['answer']) > 300 else f'  {res_a["answer"]}')
print()
print('  Order B -- Basel III LAST (recency position):')
print(f'  {res_b["answer"][:300]}...' if len(res_b['answer']) > 300 else f'  {res_b["answer"]}')
print()
print('Production guidance:')
print('  Place the most query-relevant document FIRST or LAST in the assembled context.')
print('  Avoid burying the key document in the middle of a multi-document context.')
print('  For long single documents, place the target section near the start.')

## Cell 6: Fintech -- Multi-Document Deal Analysis
**What this demonstrates**: Long-Context RAG applied to a three-document deal analysis scenario -- Basel III capital requirements, an ISDA master agreement, and a bank earnings report in context simultaneously. This simulates the due diligence workflow where a structured finance analyst needs to reason across a regulatory framework, a contractual agreement, and a counterparty's financial disclosures in a single integrated analysis.
**Expected output**: All three documents in context, a structured deal analysis answer citing each source, per-document attribution table, and a summary of why Long-Context RAG is the right pattern for this query class.

In [ ]:
# -- Multi-document deal analysis ------------------------------------------
# Scenario: a credit risk analyst is reviewing a new derivatives counterparty.
# They need to assess: (1) the counterparty's capital adequacy from the earnings
# report, (2) the collateral obligations from the ISDA agreement, and (3) whether
# those obligations are consistent with regulatory capital requirements.
# All three documents must be read together -- no single document answers the question.

DEAL_DOCS = {
    'Basel III Capital Requirements': DOCS['Basel III Capital Requirements'],
    'ISDA Master Agreement':          DOCS['ISDA Master Agreement'],
    'Pinnacle Financial Q3 Earnings': DOCS['Pinnacle Financial Q3 Earnings'],
}

DEAL_QUERY = (
    'As a credit risk analyst reviewing Pinnacle Financial as a derivatives counterparty: '
    '(1) What does the earnings report reveal about their capital adequacy relative to '
    'Basel III regulatory minimums? '
    '(2) What collateral and credit support obligations does the ISDA agreement impose? '
    '(3) Are there any inconsistencies or risks across these three documents that '
    'warrant further investigation?'
)

DEAL_SYSTEM = (
    'You are a structured finance credit risk analyst performing counterparty due diligence. '
    'You have been given three complete documents to read in full. '
    'Answer using ONLY the provided documents. '
    'Structure your answer in three numbered sections matching the three questions. '
    'For every factual claim, cite the source document and specific section or metric. '
    'Flag any gaps where the provided documents are silent on a risk dimension.'
)

# -- Confirm context budget before calling ---------------------------------
deal_ctx, deal_tokens = assemble_context(DEAL_DOCS, DEAL_QUERY)
print('Multi-document deal analysis:')
print(f'  Documents: {len(DEAL_DOCS)}')
for name in DEAL_DOCS:
    print(f'    {name:42s}  ~{estimate_tokens(DEAL_DOCS[name]):,} tokens')
print(f'  Combined context: ~{deal_tokens:,} tokens  (budget: {MAX_INPUT_TOKENS:,})')
assert deal_tokens <= MAX_INPUT_TOKENS, 'Context exceeds budget -- reduce document set'
print()

# -- Run deal analysis -----------------------------------------------------
t0 = time.perf_counter()
deal_result = long_context_query(DEAL_DOCS, DEAL_QUERY, DEAL_SYSTEM)
deal_ms = (time.perf_counter() - t0) * 1000

print('=' * 65)
print('COUNTERPARTY DUE DILIGENCE -- CREDIT RISK ANALYSIS')
print('=' * 65)
print(deal_result['answer'])
print('=' * 65)

# -- Per-document attribution summary -------------------------------------
print()
print('Per-document attribution (sources the answer draws from):')
answer_lower = deal_result['answer'].lower()
attribution_signals = {
    'Basel III Capital Requirements': ['basel', 'cet1', 'tier 1', 'capital ratio', 'leverage ratio', 'lcr'],
    'ISDA Master Agreement':          ['isda', 'collateral', 'margin', 'credit support', 'close-out', 'netting'],
    'Pinnacle Financial Q3 Earnings': ['pinnacle', 'earnings', 'q3', 'net income', 'revenue', 'nim'],
}
for doc_name, signals in attribution_signals.items():
    hits = [s for s in signals if s in answer_lower]
    status = f'cited ({len(hits)} signal(s): {", ".join(hits[:3])})' if hits else 'NOT cited -- possible lost-in-the-middle'
    print(f'  {doc_name:42s}  {status}')

print()
print(f'Query cost:  {deal_result["cost_str"]}')
print(f'Latency:     {deal_ms:.0f} ms')
print(f'Retrieval:   {deal_result["retrieval_calls"]} calls')
print()
print('Why Long-Context RAG is the right pattern for this query:')
print('  Question 1 requires reading the earnings report AND Basel III together --')
print('    a chunked retriever would have to surface the right ratio from each.')
print('  Question 2 requires the full ISDA to locate collateral terms that')
print('    reference definitions elsewhere in the same agreement.')
print('  Question 3 requires cross-document reasoning that no single retriever')
print('    can do -- the model must hold all three documents in mind simultaneously.')
print()
print('When this pattern would break down:')
print('  Add 50 more counterparty agreements -> corpus exceeds context window.')
print('  Solution: retrieve the top-3 most relevant agreements first, then')
print('  apply Long-Context RAG to those 3 -- hybrid retrieve-then-read approach.')